# RIO Trend Analysis (Climate Year)

Loads the climate-year (March 15 – March 14) re-indexed RIO dataset exported by
`RIO_daily_heatmaps.ipynb` and computes spring-opening / fall-closing trends.

Because every season is fully contained in a single row (DOY 0 = March 15,
DOY 364 = March 14 of the following year), there is no Dec/Jan year-boundary
wraparound to handle -- unlike the calendar-year version of this analysis.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import climate_year_trend_utils as cyt

## User configuration

In [ ]:
# ===== USER CONFIGURATION =====

# --- Input dataset (produced by the export cell in RIO_daily_heatmaps.ipynb) ---
route_label = 'text_route'   # must match the route_label used when the .nc was saved
nc_path     = os.path.join('.', f'RIO_climate_year_route{route_label}.nc')

# --- Vessel class to analyse (must be present in the saved dataset) ---
vessel_class = 'NIS'

# --- Output ---
save_figs  = True
output_dir = os.path.join('.', 'figures')

# --- Ensemble statistic to analyse ---
# Options: 'median', '5th_percentile', '95th_percentile'
trend_stat = 'median'

# --- RIO threshold for detecting spring/fall transitions ---
# Spring: first day RIO >= threshold (route opens as ice retreats)
# Fall:   last  day RIO >= threshold (route closes as ice advances)
trend_threshold_rio = 0

# --- Rolling-average window (days) applied before threshold detection ---
trend_rolling_window = 7

# --- DOY search windows, in SHIFTED space (0 = March 15) ---
# No wraparound syntax needed -- every range is a plain (start, end) with end > start.
# Use cyt.calendar_doy_to_shifted(calendar_doy) to translate a familiar calendar-DOY
# value into shifted space, e.g. cyt.calendar_doy_to_shifted(166) -> spring range start.
trend_spring_doy_range = (47, 108)    # roughly mid-June - mid-August (shifted)
trend_fall_doy_range   = (232, 323)   # roughly Nov - end of Jan (shifted; was the
                                       # problematic (305,365)/(350,30) calendar-year case)

# --- x-axis centering (shifted DOY to place at the centre of the heatmap x-axis) ---
# 185 (shifted) ~ Sept 16, near the annual Arctic sea-ice minimum.
plot_center_doy = 185

# --- Heatmap colour scale ---
heatmap_vmin, heatmap_vmax = -30, 30
heatmap_contours = [0, -10]

## Load the climate-year dataset

In [ ]:
ds = xr.open_dataset(nc_path)
print(ds)

if vessel_class not in ds['vessel_class'].values:
    raise ValueError(
        f"vessel_class '{vessel_class}' not found in {nc_path}; "
        f"available: {list(ds['vessel_class'].values)}"
    )

_stat_key_map = {
    'median':          'summary_rio_med',
    '5th_percentile':  'summary_rio_5th',
    '95th_percentile': 'summary_rio_95th',
}
_stat_label_map = {
    'median':          'Median',
    '5th_percentile':  '5th percentile',
    '95th_percentile': '95th percentile',
}
if trend_stat not in _stat_key_map:
    raise ValueError(
        f"trend_stat must be 'median', '5th_percentile', or '95th_percentile'; "
        f"got '{trend_stat}'"
    )
_stat_key   = _stat_key_map[trend_stat]
_stat_label = _stat_label_map[trend_stat]

data_arr     = ds[_stat_key].sel(vessel_class=vessel_class).values   # (n_seasons, 365)
season_years = [int(y) for y in ds['season_year'].values]
n_seasons    = len(season_years)
year_arr     = np.array(season_years, dtype=float)

print(f'\nLoaded {_stat_label} RIO for vessel class {vessel_class!r}: '
      f'{n_seasons} season(s), {season_years[0]}\u2013{season_years[-1] + 1}')

## Spring / fall crossing detection and trend fit

In [ ]:
smoothed = cyt.smooth_by_year(data_arr, trend_rolling_window)

spring_doys = np.full(n_seasons, np.nan)
fall_doys   = np.full(n_seasons, np.nan)
for yi in range(n_seasons):
    spring_doys[yi] = cyt.find_crossing_doy(
        smoothed[yi], trend_threshold_rio,
        trend_spring_doy_range[0], trend_spring_doy_range[1],
        find='first_above',   # RIO rises above threshold -> route opens
    )
    fall_doys[yi] = cyt.find_crossing_doy(
        smoothed[yi], trend_threshold_rio,
        trend_fall_doy_range[0], trend_fall_doy_range[1],
        find='last_above',    # RIO drops below threshold -> route closes
    )

print(f'===== {vessel_class}  ({_stat_label})  threshold = {trend_threshold_rio} =====')

print('  Spring (first day >= threshold; route opens):')
sp_slope, sp_int, sp_r2, sp_rmse, sp_yrs, sp_doys = cyt.compute_trend(spring_doys, year_arr)
if not np.isnan(sp_slope):
    print(f'    slope = {sp_slope:+.2f} days/yr,  R\u00b2 = {sp_r2:.3f},  '
          f'RMSE = {sp_rmse:.1f} days,  n = {len(sp_yrs)}')

print('  Fall (last day >= threshold; route closes):')
fa_slope, fa_int, fa_r2, fa_rmse, fa_yrs, fa_doys = cyt.compute_trend(fall_doys, year_arr)
if not np.isnan(fa_slope):
    print(f'    slope = {fa_slope:+.2f} days/yr,  R\u00b2 = {fa_r2:.3f},  '
          f'RMSE = {fa_rmse:.1f} days,  n = {len(fa_yrs)}')

## Trend scatter plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'RIO Trend (Climate Year)  \u2014  {vessel_class}  |  {_stat_label}  |  '
    f'Route {route_label}\n'
    f'Threshold = {trend_threshold_rio}   Smoothing window = {trend_rolling_window} days',
    fontsize=12, fontweight='bold',
)

_seasons = [
    ('Spring', 'First day \u2265 threshold (route opens)', spring_doys,
     sp_slope, sp_int, sp_r2, sp_rmse),
    ('Fall',   'Last day \u2265 threshold (route closes)', fall_doys,
     fa_slope, fa_int, fa_r2, fa_rmse),
]

for ax, (season_name, season_desc, all_doys, slope, intercept, r2, rmse) in zip(axes, _seasons):
    valid_mask = ~np.isnan(all_doys)
    ax.scatter(year_arr[valid_mask], all_doys[valid_mask],
               color='steelblue', s=50, zorder=3, label='Observed')
    if not np.isnan(slope):
        fit_x = np.array([year_arr[valid_mask].min(), year_arr[valid_mask].max()])
        fit_y = slope * fit_x + intercept
        ax.plot(fit_x, fit_y, 'r-', linewidth=2, label=f'Trend: {slope:+.2f} d/yr')
        stats_text = f'R\u00b2 = {r2:.3f}\nRMSE = {rmse:.1f} d'
        ax.text(0.97, 0.05, stats_text, transform=ax.transAxes,
               ha='right', va='bottom', fontsize=10,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

    if valid_mask.any():
        _pad = 14
        _ylo = max(0,   float(np.nanmin(all_doys[valid_mask])) - _pad)
        _yhi = min(364, float(np.nanmax(all_doys[valid_mask])) + _pad)
    else:
        _ylo, _yhi = 0, 364
    _vis_ticks  = [d for d in cyt.SHIFTED_MONTH_DOY if _ylo <= d <= _yhi]
    _vis_labels = [cyt.SHIFTED_MONTH_NAMES[i] for i, d in enumerate(cyt.SHIFTED_MONTH_DOY) if _ylo <= d <= _yhi]
    ax.set_yticks(_vis_ticks)
    ax.set_yticklabels(_vis_labels, fontsize=9)
    ax.set_ylim(_ylo, _yhi)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_title(f'{season_name}: {season_desc}', fontsize=11)
    ax.set_xlabel('Season-start year', fontsize=10)
    ax.set_ylabel('Day of season (shifted, 0 = Mar 15)', fontsize=10)
    ax.set_xticks(year_arr.astype(int))
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()

if save_figs:
    os.makedirs(output_dir, exist_ok=True)
    _stat_slug = _stat_label.lower().replace(' ', '_').replace('th', '').replace('st', '')
    fname = os.path.join(output_dir, f'{vessel_class}_RIO_trend_climate_year_route{route_label}_{_stat_slug}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')

plt.show()

## Heatmap with trend overlay

Each row is a full ice season (March 15 – March 14), so the fall- and
spring-crossing markers below are always exactly one per row — no double or
missing markers, unlike the calendar-year heatmap overlay.

In [ ]:
_start_doy = (plot_center_doy - 365 // 2) % 365

def _to_rolled(doy):
    return (doy - _start_doy) % 365

trend_overlay_arg = {}
for season_key, all_doys, slope, intercept in [
    ('spring', spring_doys, sp_slope, sp_int),
    ('fall',   fall_doys,   fa_slope, fa_int),
]:
    valid = ~np.isnan(all_doys)
    trend_overlay_arg[season_key] = {
        'yi':        np.where(valid)[0].astype(float),
        'rolled_x':  np.array([_to_rolled(d) for d in all_doys[valid]]),
        'slope':     slope,
        'intercept': intercept,
    }

fig, ax = cyt.plot_climate_year_heatmap_with_trend(
    data           = data_arr,
    season_years   = season_years,
    stat_label     = _stat_label,
    field_label    = 'RIO',
    cmap_name      = 'RdBu',
    vmin           = heatmap_vmin,
    vmax           = heatmap_vmax,
    cbar_label     = 'Route-summary RIO',
    center_doy     = plot_center_doy,
    contour_levels = heatmap_contours,
    trend_overlay  = trend_overlay_arg,
    route_label    = f'{route_label} ({vessel_class})',
    save_figs      = save_figs,
    output_dir     = output_dir,
    field_slug     = f'RIO_{vessel_class}',
)
plt.show()